# CardioIA – Sistema Preditivo Multiagente

Projeto de Inteligência Artificial para predição de pico de risco cardiovascular utilizando aprendizado de máquina e arquitetura multiagente.

Nesta primeira etapa, será criada uma base sintética de pacientes, treinado um modelo de classificação e realizada a avaliação de suas métricas.

Na segunda etapa, o modelo será integrado a agentes especializados para análise de risco e consulta de protocolos simulados.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 1. Geração da base sintética

Será criada uma base de dados sintética contendo características de pacientes e informações relacionadas ao sistema.

As variáveis utilizadas serão idade, frequência cardíaca, saturação de oxigênio, carga do sistema e disponibilidade de recursos.

In [2]:
np.random.seed(42)

n = 1000

idade = np.random.randint(20, 90, n)
frequencia_cardiaca = np.random.randint(50, 150, n)
spo2 = np.random.randint(85, 100, n)
carga_sistema = np.random.randint(0, 101, n)
disponibilidade_recursos = np.random.randint(0, 2, n)

## 2. Definição da variável alvo

A variável `pico_risco` representa a classificação utilizada pelo modelo.

Para esta base sintética, o risco será definido a partir de uma combinação de idade, frequência cardíaca e saturação de oxigênio.

In [3]:
pico_risco = (
    (idade > 60) &
    (frequencia_cardiaca > 100) &
    (spo2 < 94)
).astype(int)

In [4]:
dados = pd.DataFrame({
    "idade": idade,
    "frequencia_cardiaca": frequencia_cardiaca,
    "spo2": spo2,
    "carga_sistema": carga_sistema,
    "disponibilidade_recursos": disponibilidade_recursos,
    "pico_risco": pico_risco
})

dados.head()

,idade,frequencia_cardiaca,spo2,carga_sistema,disponibilidade_recursos,pico_risco
0,71,123,97,20,1,0
1,34,56,88,19,0,0
2,80,82,94,25,0,0
3,40,72,93,67,1,0
4,43,134,98,85,1,0


In [5]:
print("Quantidade de registros:", len(dados))
print("\nDistribuição do alvo:")
print(dados["pico_risco"].value_counts())

Quantidade de registros: 1000

Distribuição do alvo:
pico_risco
0    878
1    122
Name: count, dtype: int64


## 3. Preparação dos dados

A variável `pico_risco` será utilizada como variável alvo.

As demais variáveis serão utilizadas como características de entrada do modelo.

A base será dividida em dados de treinamento e dados de teste.

In [6]:
X = dados.drop("pico_risco", axis=1)
y = dados["pico_risco"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)

Treino: (800, 5)
Teste: (200, 5)


## 4. Treinamento do modelo

Será utilizado o algoritmo Random Forest Classifier para realizar a classificação dos pacientes.

O modelo será treinado utilizando os dados de treinamento.

In [7]:
modelo = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

modelo.fit(X_treino, y_treino)

print("Modelo treinado com sucesso!")

Modelo treinado com sucesso!


## 5. Avaliação do modelo

Após o treinamento, o modelo será utilizado para realizar previsões nos dados de teste.

Serão analisadas a acurácia, a matriz de confusão e o relatório de classificação.

In [8]:
previsoes = modelo.predict(X_teste)

acuracia = accuracy_score(y_teste, previsoes)

print(f"Acurácia: {acuracia:.2%}")

Acurácia: 100.00%


In [9]:
matriz = confusion_matrix(y_teste, previsoes)

print("Matriz de Confusão:")
print(matriz)

print("\nRelatório de Classificação:")
print(classification_report(y_teste, previsoes))

Matriz de Confusão:
[[176   0]
 [  0  24]]

Relatório de Classificação:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       176
           1       1.00      1.00      1.00        24

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



## 6. Simulação de um novo paciente

Nesta etapa será criado um novo paciente que não fazia parte da base de treinamento.

O modelo será utilizado para estimar a probabilidade de pico de risco e sua classificação.

In [10]:
novo_paciente = pd.DataFrame({
    "idade": [72],
    "frequencia_cardiaca": [125],
    "spo2": [90],
    "carga_sistema": [85],
    "disponibilidade_recursos": [0]
})

previsao = modelo.predict(novo_paciente)
probabilidades = modelo.predict_proba(novo_paciente)

probabilidade_risco = probabilidades[0][1]

if previsao[0] == 1:
    classificacao = "Alto risco"
else:
    classificacao = "Baixo risco"

print(f"Probabilidade de risco: {probabilidade_risco:.2%}")
print(f"Classificação: {classificacao}")

Probabilidade de risco: 100.00%
Classificação: Alto risco


## 7. Salvamento do modelo

O modelo treinado será salvo em formato `.pkl` para que possa ser utilizado posteriormente pelos agentes do sistema CardioIA.

In [12]:
import joblib

joblib.dump(modelo, "modelo_cardioia.pkl")

print("Modelo salvo com sucesso!")

Modelo salvo com sucesso!


In [13]:
!pip install -q openai-agents

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.9 MB/s eta 0:00:00


In [16]:
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [18]:
import subprocess
import time

processo_ollama = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama iniciado!")

Ollama iniciado!


In [19]:
!ollama pull qwen2.5:0.5b

## 8. Configuração do ambiente multiagente

Nesta etapa será configurado um modelo de linguagem local para executar os agentes do CardioIA.

O uso de um modelo local permite executar a arquitetura multiagente sem depender de créditos de API.

In [20]:
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool
from agents import handoff, set_tracing_disabled

set_tracing_disabled(True)

cliente_ollama = AsyncOpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama"
)

modelo_local = OpenAIChatCompletionsModel(
    model="qwen2.5:0.5b",
    openai_client=cliente_ollama
)

print("Modelo local configurado!")

Modelo local configurado!


## 9. Agente Analista de Risco

O Agente Analista de Risco é responsável por consultar o modelo preditivo treinado anteriormente.

Ele recebe os dados do paciente e utiliza uma ferramenta Python para obter a probabilidade e a classificação do risco.

In [21]:
@function_tool
def ferramenta_risco(
    idade: int,
    frequencia_cardiaca: int,
    spo2: int,
    carga_sistema: int,
    disponibilidade_recursos: int
) -> str:
    """Calcula o risco cardiovascular utilizando o modelo preditivo CardioIA."""

    paciente = pd.DataFrame({
        "idade": [idade],
        "frequencia_cardiaca": [frequencia_cardiaca],
        "spo2": [spo2],
        "carga_sistema": [carga_sistema],
        "disponibilidade_recursos": [disponibilidade_recursos]
    })

    previsao = modelo.predict(paciente)
    probabilidades = modelo.predict_proba(paciente)

    probabilidade = float(probabilidades[0][1])

    if previsao[0] == 1:
        classificacao = "Alto risco"
    else:
        classificacao = "Baixo risco"

    return (
        f"Probabilidade de pico de risco: {probabilidade:.2%}. "
        f"Classificação: {classificacao}."
    )

In [22]:
analista_risco = Agent(
    name="Analista de Risco",
    instructions="""
Você é o Agente Analista de Risco do sistema CardioIA.

Sua função é analisar os dados de um paciente.

Sempre utilize a ferramenta ferramenta_risco para consultar o modelo preditivo.

Nunca invente uma probabilidade ou classificação.

Apresente:
- probabilidade de pico de risco;
- classificação do risco.
""",
    tools=[ferramenta_risco],
    model=modelo_local
)

print("Agente Analista de Risco criado!")

Agente Analista de Risco criado!


## 10. Agente Especialista em Protocolos

O Agente Especialista em Protocolos consulta uma base simulada de protocolos médicos de acordo com a classificação de risco fornecida pelo sistema.

In [23]:
protocolos = {
    "Alto risco": [
        "Monitoramento cardíaco contínuo",
        "Avaliação médica prioritária",
        "Verificação frequente dos sinais vitais"
    ],

    "Baixo risco": [
        "Monitoramento de rotina",
        "Acompanhamento dos sinais vitais",
        "Manutenção do acompanhamento clínico"
    ]
}

In [24]:
@function_tool
def ferramenta_protocolos(classificacao: str) -> str:
    """Consulta os protocolos simulados do CardioIA."""

    if classificacao not in protocolos:
        return "Classificação de risco não encontrada."

    lista = protocolos[classificacao]

    return "Protocolos sugeridos:\n- " + "\n- ".join(lista)

In [25]:
especialista_protocolos = Agent(
    name="Especialista em Protocolos",
    instructions="""
Você é o Agente Especialista em Protocolos do CardioIA.

Sua função é consultar a base de protocolos simulados.

Utilize a ferramenta ferramenta_protocolos.

Não invente protocolos.

Apresente os protocolos correspondentes à classificação de risco recebida.
""",
    tools=[ferramenta_protocolos],
    model=modelo_local
)

print("Agente Especialista em Protocolos criado!")

Agente Especialista em Protocolos criado!


## 11. Handoffs entre os agentes

Os handoffs permitem encaminhar a execução entre agentes especializados.

O Orquestrador poderá encaminhar a análise para o Agente Analista de Risco e, posteriormente, para o Agente Especialista em Protocolos.

In [26]:
handoff_analista = handoff(
    agent=analista_risco,
    tool_name_override="encaminhar_para_analista",
    tool_description_override="Encaminha os dados do paciente para o Agente Analista de Risco."
)

handoff_protocolos = handoff(
    agent=especialista_protocolos,
    tool_name_override="encaminhar_para_protocolos",
    tool_description_override="Encaminha a classificação de risco para o Agente Especialista em Protocolos."
)

print("Handoffs configurados!")

Handoffs configurados!


## 12. Agente Orquestrador

O Agente Orquestrador coordena o fluxo do sistema.

Ele recebe os dados do paciente, encaminha a solicitação aos agentes especializados e organiza o resultado final.

In [27]:
orquestrador = Agent(
    name="Orquestrador CardioIA",
    instructions="""
Você é o Orquestrador do sistema CardioIA.

Sua função é coordenar os agentes especializados.

Primeiro, encaminhe os dados do paciente para o Agente Analista de Risco.

Depois, encaminhe o resultado da classificação para o Agente Especialista em Protocolos.

Ao final, apresente uma resposta estruturada contendo:
- probabilidade de risco;
- classificação;
- protocolos sugeridos.

Não invente informações.
""",
    handoffs=[
        handoff_analista,
        handoff_protocolos
    ],
    model=modelo_local
)

print("Orquestrador criado!")

Orquestrador criado!


In [28]:
resultado = await Runner.run(
    analista_risco,
    """
Analise o seguinte paciente:

Idade: 72
Frequência cardíaca: 125
SpO2: 90
Carga do sistema: 85
Disponibilidade de recursos: 0
"""
)

print(resultado.final_output)

O paciente se apresenta com alto risco cardiovascular. A probabilidade de pico de risco é de 100.00%.


In [33]:
especialista_protocolos = Agent(
    name="Especialista em Protocolos",
    instructions="""
Você é o Agente Especialista em Protocolos do CardioIA.

Sua função é informar os protocolos correspondentes à classificação de risco.

Para Alto risco, utilize:
- Monitoramento cardíaco contínuo
- Avaliação médica prioritária
- Verificação frequente dos sinais vitais

Para Baixo risco, utilize:
- Monitoramento de rotina
- Acompanhamento dos sinais vitais
- Manutenção do acompanhamento clínico

Não invente protocolos.
Responda de forma objetiva.
""",
    model=modelo_local
)

print("Especialista em Protocolos atualizado!")

Especialista em Protocolos atualizado!


In [42]:
resultado_final = executar_fluxo_cardioia(
    idade=72,
    frequencia_cardiaca=125,
    spo2=90,
    carga_sistema=85,
    disponibilidade_recursos=0
)

print("=== CARDIOIA ===")
print(f"Probabilidade de risco: {resultado_final['probabilidade']:.2%}")
print(f"Classificação: {resultado_final['classificacao']}")
print("\nProtocolos sugeridos:")

for protocolo in resultado_final["protocolos"]:
    print(f"- {protocolo}")

=== CARDIOIA ===
Probabilidade de risco: 100.00%
Classificação: Alto risco

Protocolos sugeridos:
- Monitoramento cardíaco contínuo
- Avaliação médica prioritária
- Verificação frequente dos sinais vitais


## 15. Integração do Agente Analista de Risco

O Agente Analista de Risco utiliza uma ferramenta conectada ao modelo preditivo para obter a probabilidade e a classificação do paciente.

Os resultados numéricos são calculados diretamente pelo modelo de Machine Learning.

In [43]:
resultado_analista = await Runner.run(
    analista_risco,
    """
Paciente:
Idade: 72
Frequência cardíaca: 125
SpO2: 90
Carga do sistema: 85
Disponibilidade de recursos: 0

Utilize a ferramenta de risco para analisar este paciente.
"""
)

print(resultado_analista.final_output)

A probabilidade de pico de risco para este paciente é de 100%. Ele é considerado alto risco.


## 16. Fluxo completo do CardioIA

O fluxo completo integra o modelo preditivo, o Agente Analista de Risco e a base de protocolos.

O sistema recebe os dados de um novo paciente, realiza a predição do risco, identifica a classificação e consulta os protocolos correspondentes.

In [45]:
await fluxo_completo_cardioia(
    idade=72,
    frequencia_cardiaca=125,
    spo2=90,
    carga_sistema=85,
    disponibilidade_recursos=0
)

=== CARDIOIA ===

ANÁLISE DE RISCO


RESULTADO DO MODELO
Probabilidade: 100.00%
Classificação: Alto risco

PROTOCOLOS SUGERIDOS
- Monitoramento cardíaco contínuo
- Avaliação médica prioritária
- Verificação frequente dos sinais vitais


In [47]:
async def fluxo_completo_cardioia(
    idade,
    frequencia_cardiaca,
    spo2,
    carga_sistema,
    disponibilidade_recursos
):
    # Resultado calculado pelo modelo de Machine Learning
    resultado_modelo = executar_fluxo_cardioia(
        idade,
        frequencia_cardiaca,
        spo2,
        carga_sistema,
        disponibilidade_recursos
    )

    # Consulta ao Agente Analista
    resultado_analista = await Runner.run(
        analista_risco,
        f"""
Paciente:
Idade: {idade}
Frequência cardíaca: {frequencia_cardiaca}
SpO2: {spo2}
Carga do sistema: {carga_sistema}
Disponibilidade de recursos: {disponibilidade_recursos}

Utilize a ferramenta de risco e informe somente:
Probabilidade: [resultado]
Classificação: [resultado]
"""
    )

    print("=== CARDIOIA ===")

    print("\nANÁLISE DE RISCO")

    if resultado_analista.final_output:
        print(resultado_analista.final_output)
    else:
        print(
            f"Probabilidade: {resultado_modelo['probabilidade']:.2%}\n"
            f"Classificação: {resultado_modelo['classificacao']}"
        )

    print("\nPROTOCOLOS SUGERIDOS")

    for protocolo in resultado_modelo["protocolos"]:
        print(f"- {protocolo}")

## 17. Conclusão

O CardioIA foi desenvolvido como um sistema preditivo multiagente capaz de utilizar um modelo de Machine Learning para classificação de risco e uma base simulada de protocolos.

O modelo Random Forest realiza a predição do pico de risco, enquanto os agentes especializados participam da análise e organização das informações.

Os resultados demonstram o funcionamento da arquitetura proposta. Entretanto, como os dados utilizados são sintéticos e a variável alvo foi definida por uma regra determinística, os resultados não representam desempenho clínico real.

Para uma aplicação real, seria necessária a utilização de dados clínicos reais e anonimizados, validação adequada, monitoramento do modelo e avaliação por profissionais especializados.

In [55]:
!ollama serve > /tmp/ollama.log 2>&1 &
!ollama list

NAME            ID              SIZE      MODIFIED       
qwen2.5:0.5b    a8b0c5157701    397 MB    32 minutes ago    


In [56]:
import time
time.sleep(3)

print("Ollama reiniciado.")

Ollama reiniciado.


In [57]:
from openai import AsyncOpenAI

cliente_ollama = AsyncOpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama"
)

print("Cliente Ollama configurado.")

Cliente Ollama configurado.


In [58]:
from agents import OpenAIChatCompletionsModel

modelo_local = OpenAIChatCompletionsModel(
    model="qwen2.5:0.5b",
    openai_client=cliente_ollama
)

print("Modelo local conectado.")

Modelo local conectado.


In [59]:
resultado_teste = await Runner.run(
    analista_risco,
    """
Paciente:
Idade: 72
Frequência cardíaca: 125
SpO2: 90
Carga do sistema: 85
Disponibilidade de recursos: 0

Utilize a ferramenta de risco e informe a probabilidade e a classificação.
"""
)

print(resultado_teste.final_output)

O probabilidade de pico de risco para o paciente está na casa da centena. Ele possui o risco de alto pela categoria A, portanto, ele deve ser frequentemente monitorado e atendido.


In [61]:
resultado_final = executar_fluxo_cardioia(
    idade=72,
    frequencia_cardiaca=125,
    spo2=90,
    carga_sistema=85,
    disponibilidade_recursos=0
)

print("=== CARDIOIA ===")
print(f"Probabilidade de risco: {resultado_final['probabilidade']:.2%}")
print(f"Classificação: {resultado_final['classificacao']}")

print("\nProtocolos sugeridos:")
for protocolo in resultado_final["protocolos"]:
    print(f"- {protocolo}")

=== CARDIOIA ===
Probabilidade de risco: 100.00%
Classificação: Alto risco

Protocolos sugeridos:
- Monitoramento cardíaco contínuo
- Avaliação médica prioritária
- Verificação frequente dos sinais vitais
